In [1]:
! pip install tensorflow

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
import sys
print(sys.version)

TensorFlow version: 2.17.0
NumPy version: 1.26.4
Pandas version: 2.2.2
3.9.19 (main, May  6 2024, 20:12:36) [MSC v.1916 64 bit (AMD64)]


In [3]:
(x_train, _), (_, _) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 127.5 - 1.0
x_train = x_train.reshape(x_train.shape[0], 784)

df = pd.DataFrame(x_train)
df.to_csv("mnist_data.csv", index=False)

data = pd.read_csv("mnist_data.csv").values

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 75s 7us/step


In [4]:
latent_dim = 100

generator = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(latent_dim,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(784, activation="tanh")
])

f:\Anaconda\envs\notebook\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
discriminator = keras.Sequential([
    layers.Dense(256, activation="relu", input_shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

discriminator.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [6]:
discriminator.trainable = False

gan_input = layers.Input(shape=(latent_dim,))
fake_image = generator(gan_input)
gan_output = discriminator(fake_image)

gan = keras.Model(gan_input, gan_output)

gan.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy"
)

In [10]:
epochs = 50
batch_size = 128
half_batch = batch_size // 2

for epoch in range(epochs):
    idx = np.random.randint(0, data.shape[0], half_batch)
    real_imgs = data[idx]

    noise = np.random.normal(0, 1, (half_batch, latent_dim))
    fake_imgs = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_imgs, np.ones((half_batch, 1)))
    d_loss_fake = discriminator.train_on_batch(fake_imgs, np.zeros((half_batch, 1)))

    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

    if epoch % 5 == 0:
        print("Epoch", epoch, "| D Loss:", d_loss_real[0], "| D Acc:", d_loss_real[1]*100, "% | G Loss:", g_loss)

Epoch 0 | D Loss: 10.786701 | D Acc: 1.9736841320991516 % | G Loss: [array(10.809713, dtype=float32), array(10.809713, dtype=float32), array(0.01861702, dtype=float32)]
Epoch 5 | D Loss: 9.127936 | D Acc: 6.924882531166077 % | G Loss: [array(9.23908, dtype=float32), array(9.23908, dtype=float32), array(0.06674208, dtype=float32)]
Epoch 10 | D Loss: 8.487544 | D Acc: 8.361774682998657 % | G Loss: [array(8.611366, dtype=float32), array(8.611366, dtype=float32), array(0.08139535, dtype=float32)]
Epoch 15 | D Loss: 8.182031 | D Acc: 9.014745056629181 % | G Loss: [array(8.29277, dtype=float32), array(8.29277, dtype=float32), array(0.08825459, dtype=float32)]
Epoch 20 | D Loss: 8.014936 | D Acc: 9.768211841583252 % | G Loss: [array(8.112844, dtype=float32), array(8.112844, dtype=float32), array(0.09598698, dtype=float32)]
Epoch 25 | D Loss: 7.9070377 | D Acc: 10.20168885588646 % | G Loss: [array(7.9925623, dtype=float32), array(7.9925623, dtype=float32), array(0.10050832, dtype=float32)]
Epo

In [11]:
idx = np.random.randint(0, data.shape[0], 1000)
real_imgs = data[idx]

noise = np.random.normal(0, 1, (1000, latent_dim))
fake_imgs = generator.predict(noise, verbose=0)

d_loss_real = discriminator.evaluate(real_imgs, np.ones((1000, 1)), verbose=0)
d_loss_fake = discriminator.evaluate(fake_imgs, np.zeros((1000, 1)), verbose=0)

final_acc = (d_loss_real[1] + d_loss_fake[1]) / 2

print("Final Discriminator Accuracy:", final_acc * 100)

Final Discriminator Accuracy: 13.40000033378601
